# Michi-Adapter — Train on Colab

Trains the Mimi ↔ Gemma projection (3 phases) on a free Colab GPU and
uploads the checkpoints to a Hugging Face Hub repo so the local macOS
app can pull them.

- **Phase 1** — synthetic alignment (warmup, ~2 min on T4)
- **Phase 2** — real audio via DailyTalkContiguous → Mimi codec (~10 min)
- **Phase 3** — supervised SFT with conversational text (~15 min)

Outputs:
```
adapter_phase1.pt
adapter_phase2.pt
adapter_phase3_sft.pt
```
pushed to `lucahttp/michi-adapter-checkpoints` on HF Hub (private).

## 0. Auth

In [ ]:
import os, subprocess, sys
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')  # set in Colab Secrets (left panel 🔑)
assert HF_TOKEN, 'Add HF_TOKEN to Colab Secrets first'
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
print('HF_TOKEN ok (len=%d)' % len(HF_TOKEN))

In [ ]:
# Clone michifus
REPO = 'lucahttp/michifus'
WORKDIR = '/content/michifus'
if not os.path.exists(WORKDIR):
    subprocess.check_call(['git', 'clone', '--depth', '1',
                            f'https://github.com/{REPO}.git', WORKDIR])
%cd {WORKDIR}
!ls

In [ ]:
# Install deps
!pip install -q -U pip
!pip install -q torch torchvision torchaudio transformers accelerate \
    datasets sentencepiece huggingface_hub[cli] hf_transfer moshi 'numpy<2' soundfile

## 1. Phase 1 — synthetic alignment

In [ ]:
!python train.py --steps 200 --batch_size 32 --save_path checkpoints/adapter_phase1.pt

## 2. Phase 2 — real audio alignment

In [ ]:
!python train_phase2.py --steps 800 --batch_size 16 \
    --phase1_ckpt checkpoints/adapter_phase1.pt \
    --save_path checkpoints/adapter_phase2.pt

## 3. Phase 3 — supervised fine-tune

In [ ]:
!python train_phase3_sft.py --steps 400 --batch_size 8 \
    --phase2_ckpt checkpoints/adapter_phase2.pt \
    --save_path checkpoints/adapter_phase3_sft.pt

## 4. Upload checkpoints to HF Hub

In [ ]:
from huggingface_hub import HfApi
api = HfApi(token=HF_TOKEN)
REPO_ID = 'lucahttp/michi-adapter-checkpoints'
api.create_repo(REPO_ID, repo_type='model', private=True, exist_ok=True)
for f in ['adapter_phase1.pt', 'adapter_phase2.pt', 'adapter_phase3_sft.pt']:
    p = f'checkpoints/{f}'
    if os.path.exists(p):
        api.upload_file(path_or_fileobj=p, path_in_repo=f, repo_id=REPO_ID)
        print('uploaded', f)
print('done. https://huggingface.co/' + REPO_ID)